In [1]:
import sys
import os
import urllib3
from configparser import ConfigParser

# Add your local ThreatConnect SDK to path
sys.path.append("Z:/HTOC/Data_Analytics/threatconnect")
from ThreatConnect import ThreatConnect
from RequestObject import RequestObject
from Owners import Owners

# Add your project repo to path
project_root = r"C:\Users\jaskew\Documents\project_repository\scripts\Data Movement\ThrearConnect-api-pull"
if project_root not in sys.path:
    sys.path.append(project_root)

from utils.config_loader import load_config

# Load API config
config_path = os.path.join(project_root, "utils", "config.json")
try:
    api_secret_key, api_access_id, api_base_url, api_default_org = load_config(config_path)
    print(f"Loaded config from: {config_path}")
    print(f"Base URL: {api_base_url}")
    print(f"Access ID: {api_access_id}")
    print(f"Default Org: {api_default_org}")
except Exception as e:
    print(f"[ERROR] Failed to load configuration: {e}")
    sys.exit(1)

# Disable SSL verification warnings (use cautiously)
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
verify_ssl = False

# Initialize ThreatConnect session
try:
    tc = ThreatConnect(api_access_id, api_secret_key, api_default_org, api_base_url)
    print("ThreatConnect initialized.")
except Exception as e:
    print(f"[ERROR] Failed to initialize ThreatConnect: {e}")
    sys.exit(1)

# Define the owner (organization scope)
owner = 'HTOC Org'

# Create a request object to fetch indicators (or other data)
try:
    ro = RequestObject()
    ro.set_http_method('GET')
    ro.set_owner(owner)
    ro.set_owner_allowed(True)
    # ro.set_resource_pagination(True)  # Uncomment if needed
    print("RequestObject successfully created.")
except Exception as e:
    print(f"[ERROR] Failed to initialize RequestObject: {e}")
    sys.exit(1)



Loaded config from: C:\Users\jaskew\Documents\project_repository\scripts\Data Movement\ThrearConnect-api-pull\utils\config.json
Base URL: https://hvs.threatconnect.com/api
Access ID: 09783848890162390382
Default Org: HTOC Org
ThreatConnect initialized.
RequestObject successfully created.


In [2]:
import pandas as pd
from datetime import datetime, timedelta
import urllib.parse
import pytz
from collections import defaultdict

# --- ThreatConnect setup (assumes 'tc' and 'ro' objects already exist) ---
# Make sure tc and ro are defined in your environment based on your SDK usage

# Define the time window (3 days ago at 00:00 UTC)
start_date = (datetime.now(pytz.UTC) - timedelta(days=3)).date()
start = f"{start_date}T00:00:00Z"

# Indicator types to query
type_names = [
    "Address", "EmailAddress", "File", "Host", "URL", "ASN", "CIDR",
    "Email Subject", "Hashtag", "Mutex", "Registry Key", "Stripped URL", "User Agent"
]
type_name_condition = ", ".join([f'"{t}"' for t in type_names])

# Owners to pull from
list_of_owners = ['HTOC Org']
final_results = []

for owner in list_of_owners:
    print(f"\nQuerying owner: {owner}")
    try:
        tql_raw = (
            f'ownerName EQ "{owner}" AND '
            f'typeName IN ({type_name_condition}) AND '
            f'lastObserved >= "{start}"'
        )
        tql_encoded = urllib.parse.quote(tql_raw)

        ro.set_http_method('GET')
        ro.set_request_uri(
            f'/v3/indicators?tql={tql_encoded}&fields=tags,observations&resultStart=0&resultLimit=10000'
        )

        response = tc.api_request(ro)

        if response.headers.get('content-type') == 'application/json':
            results = response.json()
            final_results.append(results)
        else:
            print(f"Unexpected response format: {response.headers.get('content-type')}")
    except Exception as e:
        print(f"Failed to query indicators for {owner}: {e}")

# Normalize and clean the indicators
if final_results:
    normalized_data = []
    for result in final_results:
        if 'data' in result:
            for item in result['data']:
                if 'summary' in item:
                    normalized_data.append(item)

    if normalized_data:
        observed_src = pd.json_normalize(normalized_data)
        observed_src['indicator'] = observed_src['summary'].str.split(' ', expand=True)[0].str.strip()
        observed_src = observed_src.drop_duplicates(subset='indicator', keep='first')

        # Ensure lastObserved is a datetime object for filtering
        observed_src['lastObserved'] = pd.to_datetime(observed_src['lastObserved'], utc=True)
        start_dt = pd.to_datetime(start)
        observed_src = observed_src[observed_src['lastObserved'] >= start_dt]

        print(f"\nRetrieved {len(observed_src)} unique observed indicators.")

        # -------------------------------
        # Enrich indicators with VirusTotalV3
        # -------------------------------
        indicator_ids = observed_src['id'].dropna().unique().tolist()
        enriched_results = []

        print(f"\nEnriching {len(indicator_ids)} indicators with VirusTotalV3...")

        for indicator_id in indicator_ids:
            try:
                enrich_url = f'/v3/indicators/{indicator_id}/enrich?type=VirusTotalV3'
                ro.set_http_method('POST')
                ro.set_request_uri(enrich_url)
                ro.set_body({})  # Enrichment call usually has no body

                enrich_response = tc.api_request(ro)

                if enrich_response.status_code == 200:
                    enrich_data = enrich_response.json()
                    enrich_data['indicator_id'] = indicator_id
                    enriched_results.append(enrich_data)
                else:
                    print(f"Enrichment failed for ID {indicator_id} (status {enrich_response.status_code})")
            except Exception as e:
                print(f"Error enriching indicator {indicator_id}: {e}")

        # Convert enriched data into DataFrame
        if enriched_results:
            df_enriched = pd.json_normalize(enriched_results)
            df_enriched.rename(columns={'indicator_id': 'id'}, inplace=True)

            # Merge enrichment data into main dataframe
            observed_src = observed_src.merge(df_enriched, on='id', how='left')
            print(f"\nSuccessfully enriched and merged {len(df_enriched)} indicators.")
        else:
            print("\nNo enrichment data retrieved.")

    else:
        print("\nNo valid indicators with 'summary' key retrieved.")
else:
    print("\nNo indicators retrieved.")

# Optional: Save the result
# observed_src.to_csv("observed_enriched_indicators.csv", index=False)



Querying owner: HTOC Org

Retrieved 936 unique observed indicators.

Enriching 936 indicators with VirusTotalV3...


Status Code: 400
Failed API Response: b'{"message":"Unable to enrich this indicator. Either the indicator or enrichment types are not compatible, or some other kind of error occurred while trying to get the enrichment information.","status":"Error"}'


Error enriching indicator 5629499554313220: b'{"message":"Unable to enrich this indicator. Either the indicator or enrichment types are not compatible, or some other kind of error occurred while trying to get the enrichment information.","status":"Error"}'


Status Code: 400
Failed API Response: b'{"errCode":"0x1001","message":"The Stripped URL denverite.com cannot be enriched with VirusTotal because the indicator type isn\'t supported.","status":"Error"}'
Status Code: 400
Failed API Response: b'{"errCode":"0x1001","message":"The Stripped URL chaturbate.com cannot be enriched with VirusTotal because the indicator type isn\'t supported.","status":"Error"}'
Status Code: 400
Failed API Response: b'{"errCode":"0x1001","message":"The Stripped URL fmovies.co cannot be enriched with VirusTotal because the indicator type isn\'t supported.","status":"Error"}'
Status Code: 400
Failed API Response: b'{"errCode":"0x1001","message":"The Stripped URL moplay.org cannot be enriched with VirusTotal because the indicator type isn\'t supported.","status":"Error"}'


Error enriching indicator 5629499542019877: b'{"errCode":"0x1001","message":"The Stripped URL denverite.com cannot be enriched with VirusTotal because the indicator type isn\'t supported.","status":"Error"}'
Error enriching indicator 5629499542023970: b'{"errCode":"0x1001","message":"The Stripped URL chaturbate.com cannot be enriched with VirusTotal because the indicator type isn\'t supported.","status":"Error"}'
Error enriching indicator 5629499542023958: b'{"errCode":"0x1001","message":"The Stripped URL fmovies.co cannot be enriched with VirusTotal because the indicator type isn\'t supported.","status":"Error"}'
Error enriching indicator 5629499542023960: b'{"errCode":"0x1001","message":"The Stripped URL moplay.org cannot be enriched with VirusTotal because the indicator type isn\'t supported.","status":"Error"}'


...

Error enriching indicator 5182076: b'{"errCode":"0x1001","message":"The Stripped URL roozaneh.net cannot be enriched with VirusTotal because the indicator type isn\'t supported.","status":"Error"}'
Error enriching indicator 5182090: b'{"errCode":"0x1001","message":"The Stripped URL minkch.com cannot be enriched with VirusTotal because the indicator type isn\'t supported.","status":"Error"}'
Error enriching indicator 5182091: b'{"errCode":"0x1001","message":"The Stripped URL pydata.org cannot be enriched with VirusTotal because the indicator type isn\'t supported.","status":"Error"}'
Error enriching indicator 4892914: b'{"errCode":"0x1001","message":"The EmailAddress k.baker@positiveproximity.com cannot be enriched with VirusTotal because the indicator type isn\'t supported.","status":"Error"}'


Status Code: 400
Failed API Response: b'{"errCode":"0x1001","message":"The Stripped URL onestart.ai/ cannot be enriched with VirusTotal because the indicator type isn\'t supported.","status":"Error"}'
Status Code: 400
Failed API Response: b'{"errCode":"0x1001","message":"The Stripped URL www.shorturl.at/ cannot be enriched with VirusTotal because the indicator type isn\'t supported.","status":"Error"}'
Status Code: 400
Failed API Response: b'{"errCode":"0x1001","message":"The Stripped URL geo.netsupportsoftware.com/location/loca.asp cannot be enriched with VirusTotal because the indicator type isn\'t supported.","status":"Error"}'
Status Code: 400
Failed API Response: b'{"errCode":"0x1001","message":"The Stripped URL christianityreport.net cannot be enriched with VirusTotal because the indicator type isn\'t supported.","status":"Error"}'


Error enriching indicator 6755399442005035: b'{"errCode":"0x1001","message":"The Stripped URL onestart.ai/ cannot be enriched with VirusTotal because the indicator type isn\'t supported.","status":"Error"}'
Error enriching indicator 4883044: b'{"errCode":"0x1001","message":"The Stripped URL www.shorturl.at/ cannot be enriched with VirusTotal because the indicator type isn\'t supported.","status":"Error"}'
Error enriching indicator 4324196: b'{"errCode":"0x1001","message":"The Stripped URL geo.netsupportsoftware.com/location/loca.asp cannot be enriched with VirusTotal because the indicator type isn\'t supported.","status":"Error"}'
Error enriching indicator 4585198: b'{"errCode":"0x1001","message":"The Stripped URL christianityreport.net cannot be enriched with VirusTotal because the indicator type isn\'t supported.","status":"Error"}'


Status Code: 400
Failed API Response: b'{"errCode":"0x1001","message":"The Stripped URL google cannot be enriched with VirusTotal because the indicator type isn\'t supported.","status":"Error"}'
Status Code: 400
Failed API Response: b'{"errCode":"0x1001","message":"The Stripped URL shorturl.asia cannot be enriched with VirusTotal because the indicator type isn\'t supported.","status":"Error"}'
Status Code: 400
Failed API Response: b'{"errCode":"0x1001","message":"The EmailAddress alsumood@emirates.net.ae cannot be enriched with VirusTotal because the indicator type isn\'t supported.","status":"Error"}'


Error enriching indicator 4442727: b'{"errCode":"0x1001","message":"The Stripped URL google cannot be enriched with VirusTotal because the indicator type isn\'t supported.","status":"Error"}'
Error enriching indicator 4512619: b'{"errCode":"0x1001","message":"The Stripped URL shorturl.asia cannot be enriched with VirusTotal because the indicator type isn\'t supported.","status":"Error"}'
Error enriching indicator 3671024: b'{"errCode":"0x1001","message":"The EmailAddress alsumood@emirates.net.ae cannot be enriched with VirusTotal because the indicator type isn\'t supported.","status":"Error"}'

Successfully enriched and merged 915 indicators.


In [3]:
observed_src

,id,dateAdded,ownerId,ownerName,webLink,type,lastModified,rating,confidence,description,...,data.privateFlag,data.active,data.activeLocked,data.ip,data.legacyLink,data.enrichment.data,data.hostName,data.dnsActive,data.whoisActive,data.source
0,5629499555060720,2025-06-16T17:42:49Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-06-18T12:02:29Z,3.0,80,RITM0589984,...,False,True,False,45.138.16.230,https://hvs.threatconnect.com/auth/indicators/...,"[{'type': 'VirusTotal', 'vtMaliciousCount': 13}]",NaN,NaN,NaN,NaN
1,5629499555060740,2025-06-16T17:42:59Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-06-18T12:02:07Z,3.0,80,RITM0589984,...,False,True,False,94.16.115.121,https://hvs.threatconnect.com/auth/indicators/...,"[{'type': 'VirusTotal', 'vtMaliciousCount': 5}]",NaN,NaN,NaN,NaN
2,5272597,2025-01-31T11:46:35Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-06-18T11:23:06Z,2.0,7,Executive Summary\r\nInsikt Group has identifi...,...,False,True,False,NaN,https://hvs.threatconnect.com/auth/indicators/...,"[{'type': 'VirusTotal', 'vtMaliciousCount': 0}]",jewelryexchange.com,False,False,NaN
3,5271606,2025-01-29T16:27:10Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-06-18T11:23:06Z,3.0,86,The following list of urls and domains are cur...,...,False,True,False,NaN,https://hvs.threatconnect.com/auth/indicators/...,"[{'type': 'VirusTotal', 'vtMaliciousCount': 0}]",api-docs.deepseek.com,False,False,NaN
4,5629499537014058,2025-04-20T17:39:55Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-06-18T11:23:05Z,3.0,55,Domain associated with Legion Loader and mali...,...,False,True,False,NaN,https://hvs.threatconnect.com/auth/indicators/...,"[{'type': 'VirusTotal', 'vtMaliciousCount': 0}]",uploads-ssl.webflow.com,False,False,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
931,4324196,2023-04-21T14:22:00Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Stripped URL,2025-01-16T23:23:57Z,3.0,84,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
932,4585198,2024-04-30T15:02:57Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Stripped URL,2024-04-30T15:24:35Z,4.0,51,VA users received email messages from <Good Ch...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
933,4442727,2023-10-06T23:26:48Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Stripped URL,2024-03-28T23:23:43Z,3.0,83,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
934,4512619,2024-01-26T20:41:51Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Stripped URL,2024-01-28T12:55:28Z,3.0,70,"On December 28, 2023, a VA user received and r...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
